# JAX → Jaxpr → StableHLO Practice Notebook

This notebook is a hands-on practice notebook for understanding how JAX programs are traced, represented as **Jaxpr**, lowered to **StableHLO**, and eventually compiled by XLA.

The goal is not to learn compiler theory first. The goal is to build the habit of asking:

```text
What does this JAX code become after tracing?
What does it become after lowering?
What performance implications can I infer?
```

## Learning path

```text
JAX code
  → Jaxpr
  → StableHLO / MLIR
  → XLA compilation
  → device executable
```

In this notebook, we mainly inspect:

```text
JAX code → Jaxpr → StableHLO
```

## Edited version notes

This version keeps the implementation unchanged and adds English explanatory comments
to the code cells. The comments emphasize the main insights from the walkthrough:

```text
JAX code is not executed as arbitrary Python.
Traceable JAX computations are converted into Jaxpr.
Jaxpr is lowered to StableHLO/HLO.
The compiler keeps the computations needed for the final output and can remove
dead intermediate work.
vmap is a function transformation, not a Python loop.
scan expresses recurrent computation as compiler-visible loop IR.
GNN-style message passing appears as gather + scatter-add.
```


## 0. Environment setup

This notebook assumes JAX is installed. The helper functions below print:

1. Jaxpr
2. StableHLO
3. Compiled result

The key inspection API is:

```python
jax.jit(f).lower(*args).compiler_ir(dialect="stablehlo")
```

In [4]:
# -----------------------------------------------------------------------------
# Cell 0: Environment setup
# -----------------------------------------------------------------------------
# This cell imports JAX and prints the JAX version and available devices.
#
# Main idea:
#   JAX works by tracing JAX-compatible numerical code into Jaxpr, lowering it to
#   StableHLO/HLO, and then letting XLA compile the result for CPU/GPU/TPU.
#
# Pure Python is too dynamic to generally follow this route. JAX succeeds when
# the computation is expressed through traceable JAX primitives such as jnp and lax.
# -----------------------------------------------------------------------------
import jax
import jax.numpy as jnp
from jax import lax

print("JAX version:", jax.__version__)
print("Available devices:", jax.devices())

JAX version: 0.7.2
Available devices: [CpuDevice(id=0)]


In [5]:
# -----------------------------------------------------------------------------
# Helper: inspect_jax
# -----------------------------------------------------------------------------
# This helper inspects the same JAX function from three perspectives:
#
#   1. Jaxpr
#      A relatively high-level, JAX-specific intermediate representation produced
#      by tracing the Python function.
#
#   2. StableHLO
#      A lower-level IR where shapes, dtypes, broadcasts, control flow, gather,
#      scatter, and dot_general are more explicit.
#
#   3. Compiled result
#      The actual numerical result produced by jax.jit(f)(*args).
#
# Key observation:
#   JAX is not simply "making Python faster." It is moving traceable numerical
#   computation out of Python into compiler IR.
# -----------------------------------------------------------------------------
def inspect_jax(f, *args, name=None, show_result=True):
    # Print Jaxpr, StableHLO, and optionally the result of a JAX function.
    if name:
        print(f"{'=' * 80}{name}{'=' * 80}")
        print("==== Jaxpr ====")
    try:
        print(jax.make_jaxpr(f)(*args))
    except Exception as e:
        print("[Jaxpr error]", repr(e))

    print("==== StableHLO ====")
    try:
        lowered = jax.jit(f).lower(*args)
        print(lowered.compiler_ir(dialect="stablehlo"))
    except Exception as e:
        print("[StableHLO error]", repr(e))

    if show_result:
        print("==== Result ====")
        try:
            print(jax.jit(f)(*args))
        except Exception as e:
            print("[Execution error]", repr(e))

## 1. Elementwise operations

Expected mapping:

```text
jnp.sin(x)  → stablehlo.sine
x + y       → stablehlo.add
y * 2       → stablehlo.multiply
```

Focus on how scalar constants such as `2.0` appear in the IR.

In [6]:
# -----------------------------------------------------------------------------
# 1. Elementwise operations
# -----------------------------------------------------------------------------
# This cell shows how a simple high-level expression:
#
#   jnp.sin(x) + y * 2.0
#
# appears as elementary operations in Jaxpr and StableHLO.
#
# Expected mapping:
#   jnp.sin(x) -> sin / stablehlo.sine
#   y * 2.0   -> multiply, with scalar 2.0 broadcast to the vector shape
#   +         -> add
#
# Key observation:
#   A single mathematical expression in JAX becomes a shape-explicit sequence of
#   primitive operations in StableHLO.
# -----------------------------------------------------------------------------
def f_elementwise(x, y):
    return jnp.sin(x) + y * 2.0

x = jnp.ones((3,), dtype=jnp.float32)
y = jnp.arange(3, dtype=jnp.float32)

inspect_jax(f_elementwise, x, y, name="1. Elementwise: sin + multiply + add")

================================================================================1. Elementwise: sin + multiply + add================================================================================
==== Jaxpr ====
{ lambda ; a:f32[3] b:f32[3]. let
    c:f32[3] = sin a
    d:f32[3] = mul b 2.0:f32[]
    e:f32[3] = add c d
  in (e,) }
==== StableHLO ====
module @jit_f_elementwise attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<3xf32>, %arg1: tensor<3xf32>) -> (tensor<3xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.sine %arg0 : tensor<3xf32>
    %cst = stablehlo.constant dense<2.000000e+00> : tensor<f32>
    %1 = stablehlo.broadcast_in_dim %cst, dims = [] : (tensor<f32>) -> tensor<3xf32>
    %2 = stablehlo.multiply %arg1, %1 : tensor<3xf32>
    %3 = stablehlo.add %0, %2 : tensor<3xf32>
    return %3 : tensor<3xf32>
  }
}

==== Result ====
[0.84147096 2.841471   4.8414707 ]


### Exercise 1

Modify the function and observe which StableHLO operations appear.

Try: `jnp.exp`, `jnp.log`, `jnp.tanh`, division, or chained elementwise expressions.

In [7]:
# -----------------------------------------------------------------------------
# Exercise 1: Custom elementwise function
# -----------------------------------------------------------------------------
# This exercise replaces sin/multiply/add with exp/add/divide:
#
#   exp(x) / (y + 1.0)
#
# Expected mapping:
#   jnp.exp(x) -> exp / stablehlo.exponential
#   y + 1.0    -> add, with scalar 1.0 broadcast to shape (3,)
#   /          -> divide
#
# Key observation:
#   Changing the mathematical expression changes the primitive operations in
#   Jaxpr/StableHLO in a direct and readable way.
# -----------------------------------------------------------------------------
def f_elementwise_exercise(x, y):
    # TODO: modify this expression
    return jnp.exp(x) / (y + 1.0)

inspect_jax(f_elementwise_exercise, x, y, name="Exercise 1: custom elementwise function")

================================================================================Exercise 1: custom elementwise function================================================================================
==== Jaxpr ====
{ lambda ; a:f32[3] b:f32[3]. let
    c:f32[3] = exp a
    d:f32[3] = add b 1.0:f32[]
    e:f32[3] = div c d
  in (e,) }
==== StableHLO ====
module @jit_f_elementwise_exercise attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<3xf32>, %arg1: tensor<3xf32>) -> (tensor<3xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.exponential %arg0 : tensor<3xf32>
    %cst = stablehlo.constant dense<1.000000e+00> : tensor<f32>
    %1 = stablehlo.broadcast_in_dim %cst, dims = [] : (tensor<f32>) -> tensor<3xf32>
    %2 = stablehlo.add %arg1, %1 : tensor<3xf32>
    %3 = stablehlo.divide %0, %2 : tensor<3xf32>
    return %3 : tensor<3xf32>
  }
}

==== Result ====
[2.7182817 1.3591409 0.9060939]


## 2. Broadcasting

Broadcasting is essential when reading HLO.

```python
x.shape == (4, 3)
b.shape == (3,)
x + b
```

Look for:

```text
stablehlo.broadcast_in_dim
```

In [8]:
# -----------------------------------------------------------------------------
# 2. Broadcasting
# -----------------------------------------------------------------------------
# This cell shows how JAX handles x + b when:
#
#   x.shape == (4, 3)
#   b.shape == (3,)
#
# Conceptually, b is treated like a row vector and expanded across the first axis:
#
#   (3,) -> (1, 3) -> (4, 3)
#
# StableHLO makes this shape alignment explicit with stablehlo.broadcast_in_dim.
#
# Key observation:
#   Broadcasting is implicit in JAX/NumPy syntax, but explicit in HLO-like IR.
# -----------------------------------------------------------------------------
def f_broadcast(x, b):
    return x + b

x2 = jnp.ones((4, 3), dtype=jnp.float32)
b2 = jnp.arange(3, dtype=jnp.float32)

inspect_jax(f_broadcast, x2, b2, name="2. Broadcasting: (4, 3) + (3,)")

================================================================================2. Broadcasting: (4, 3) + (3,)================================================================================
==== Jaxpr ====
{ lambda ; a:f32[4,3] b:f32[3]. let
    c:f32[1,3] = broadcast_in_dim[
      broadcast_dimensions=(1,)
      shape=(1, 3)
      sharding=None
    ] b
    d:f32[4,3] = add a c
  in (d,) }
==== StableHLO ====
module @jit_f_broadcast attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4x3xf32>, %arg1: tensor<3xf32>) -> (tensor<4x3xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.broadcast_in_dim %arg1, dims = [1] : (tensor<3xf32>) -> tensor<1x3xf32>
    %1 = stablehlo.broadcast_in_dim %0, dims = [0, 1] : (tensor<1x3xf32>) -> tensor<4x3xf32>
    %2 = stablehlo.add %arg0, %1 : tensor<4x3xf32>
    return %2 : tensor<4x3xf32>
  }
}

==== Result ====
[[1. 2. 3.]
 [1. 2. 3.]
 [1. 2. 3.]
 [1. 2. 3.]]


### Exercise 2

Change `b2_ex` to shapes such as `(1, 3)`, `(4, 1)`, or scalar `()` and inspect the output.

In [9]:
# -----------------------------------------------------------------------------
# Exercise 2: Different broadcast shape
# -----------------------------------------------------------------------------
# Here b already has shape (1, 3), so it only needs to be expanded along the row
# dimension:
#
#   (1, 3) -> (4, 3)
#
# Try changing b2_ex to:
#   (4, 1)  -> column-wise broadcasting
#   scalar  -> scalar-to-matrix broadcasting
#
# Key observation:
#   To read broadcast_in_dim, always ask:
#     1. What is the input shape?
#     2. What is the output shape?
#     3. Which input axes map to which output axes?
# -----------------------------------------------------------------------------
b2_ex = jnp.ones((1, 3), dtype=jnp.float32)
inspect_jax(f_broadcast, x2, b2_ex, name="Exercise 2: different broadcast shape")

================================================================================Exercise 2: different broadcast shape================================================================================
==== Jaxpr ====
{ lambda ; a:f32[4,3] b:f32[1,3]. let c:f32[4,3] = add a b in (c,) }
==== StableHLO ====
module @jit_f_broadcast attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4x3xf32>, %arg1: tensor<1x3xf32>) -> (tensor<4x3xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.broadcast_in_dim %arg1, dims = [0, 1] : (tensor<1x3xf32>) -> tensor<4x3xf32>
    %1 = stablehlo.add %arg0, %0 : tensor<4x3xf32>
    return %1 : tensor<4x3xf32>
  }
}

==== Result ====
[[2. 2. 2.]
 [2. 2. 2.]
 [2. 2. 2.]
 [2. 2. 2.]]


## 3. Reshape and transpose

Many high-level tensor operations become combinations of:

```text
reshape
transpose
broadcast
slice
concatenate
```

Transformer and V-JEPA-like pipelines often contain many shape transformations around `dot_general`.

In [10]:
# -----------------------------------------------------------------------------
# 3. Reshape and transpose
# -----------------------------------------------------------------------------
# This cell follows a pure shape transformation:
#
#   (24,) -> reshape -> (2, 3, 4)
#          -> transpose(1, 0, 2) -> (3, 2, 4)
#          -> reshape -> (3, 8)
#
# reshape changes the view/shape while preserving the number of elements.
# transpose changes the order of axes.
#
# Key observation:
#   Transformer, V-JEPA-like, and attention pipelines often contain many reshape
#   and transpose operations around dot_general.
# -----------------------------------------------------------------------------
def f_shape_ops(x):
    x = jnp.reshape(x, (2, 3, 4))
    x = jnp.transpose(x, (1, 0, 2))
    return jnp.reshape(x, (3, 8))

x3 = jnp.arange(24, dtype=jnp.float32)
inspect_jax(f_shape_ops, x3, name="3. Shape ops: reshape + transpose + reshape")

================================================================================3. Shape ops: reshape + transpose + reshape================================================================================
==== Jaxpr ====
{ lambda ; a:f32[24]. let
    b:f32[2,3,4] = reshape[dimensions=None new_sizes=(2, 3, 4) sharding=None] a
    c:f32[3,2,4] = transpose[permutation=(1, 0, 2)] b
    d:f32[3,8] = reshape[dimensions=None new_sizes=(3, 8) sharding=None] c
  in (d,) }
==== StableHLO ====
module @jit_f_shape_ops attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<24xf32>) -> (tensor<3x8xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.reshape %arg0 : (tensor<24xf32>) -> tensor<2x3x4xf32>
    %1 = stablehlo.transpose %0, dims = [1, 0, 2] : (tensor<2x3x4xf32>) -> tensor<3x2x4xf32>
    %2 = stablehlo.reshape %1 : (tensor<3x2x4xf32>) -> tensor<3x8xf32>
    return %2 : tensor<3x8xf32>
  }
}

==== Result ====
[[ 0.  1.  2.  3. 12. 

### Exercise 3

Add another transpose or reshape. Check whether the StableHLO gets more complex or stays simple.

In [11]:
# -----------------------------------------------------------------------------
# Exercise 3: Alternative transpose
# -----------------------------------------------------------------------------
# This exercise changes the permutation to:
#
#   transpose(2, 1, 0)
#
# Shape flow:
#   (24,) -> (2, 3, 4) -> (4, 3, 2)
#
# Key observation:
#   The transpose permutation describes the order of old axes in the new tensor.
#   For permutation (2, 1, 0):
#     new axis 0 <- old axis 2
#     new axis 1 <- old axis 1
#     new axis 2 <- old axis 0
# -----------------------------------------------------------------------------
def f_shape_ops_exercise(x):
    # TODO: modify this chain
    x = jnp.reshape(x, (2, 3, 4))
    x = jnp.transpose(x, (2, 1, 0))
    return x

inspect_jax(f_shape_ops_exercise, x3, name="Exercise 3: custom shape ops")

================================================================================Exercise 3: custom shape ops================================================================================
==== Jaxpr ====
{ lambda ; a:f32[24]. let
    b:f32[2,3,4] = reshape[dimensions=None new_sizes=(2, 3, 4) sharding=None] a
    c:f32[4,3,2] = transpose[permutation=(2, 1, 0)] b
  in (c,) }
==== StableHLO ====
module @jit_f_shape_ops_exercise attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<24xf32>) -> (tensor<4x3x2xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.reshape %arg0 : (tensor<24xf32>) -> tensor<2x3x4xf32>
    %1 = stablehlo.transpose %0, dims = [2, 1, 0] : (tensor<2x3x4xf32>) -> tensor<4x3x2xf32>
    return %1 : tensor<4x3x2xf32>
  }
}

==== Result ====
[[[ 0. 12.]
  [ 4. 16.]
  [ 8. 20.]]

 [[ 1. 13.]
  [ 5. 17.]
  [ 9. 21.]]

 [[ 2. 14.]
  [ 6. 18.]
  [10. 22.]]

 [[ 3. 15.]
  [ 7. 19.]
  [11. 23.]]]


## 4. Matrix multiplication and `dot_general`

JAX matrix multiplication usually lowers to:

```text
stablehlo.dot_general
```

Many operations are variations of `dot_general`: `matmul`, `einsum`, `tensordot`, batched matmul, and attention scores.

In [12]:
# -----------------------------------------------------------------------------
# 4. Matrix multiplication and dot_general
# -----------------------------------------------------------------------------
# This cell shows that ordinary matrix multiplication lowers to dot_general.
#
# Shape flow:
#   x: (8, 16)
#   w: (16, 32)
#   x @ w -> (8, 32)
#
# In StableHLO, dot_general records which dimensions are contracted.
#
# Key observation:
#   dot_general is the central IR operation behind matmul, many einsum patterns,
#   attention scores, and dense neural network layers.
# -----------------------------------------------------------------------------
def f_matmul(x, w):
    return x @ w

x4 = jnp.ones((8, 16), dtype=jnp.float32)
w4 = jnp.ones((16, 32), dtype=jnp.float32)
inspect_jax(f_matmul, x4, w4, name="4. Matmul: x @ w")

================================================================================4. Matmul: x @ w================================================================================
==== Jaxpr ====
{ lambda ; a:f32[8,16] b:f32[16,32]. let
    c:f32[8,32] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] a b
  in (c,) }
==== StableHLO ====
module @jit_f_matmul attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<8x16xf32>, %arg1: tensor<16x32xf32>) -> (tensor<8x32xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.dot_general %arg0, %arg1, contracting_dims = [1] x [0], precision = [DEFAULT, DEFAULT] : (tensor<8x16xf32>, tensor<16x32xf32>) -> tensor<8x32xf32>
    return %0 : tensor<8x32xf32>
  }
}

==== Result ====
[[16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.
  16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]
 [16. 16. 16. 16. 16. 16. 16.

### Exercise 4: simplified attention score

Inspect:

```python
scores = q @ k.T / sqrt(d)
```

Look for `dot_general`, `transpose`, and scale operations.

In [13]:
# -----------------------------------------------------------------------------
# Exercise 4: Simplified attention score
# -----------------------------------------------------------------------------
# This is the score part of scaled dot-product attention:
#
#   scores = Q K^T / sqrt(d)
#
# Shape flow:
#   q:   (4, 16)
#   k:   (6, 16)
#   k.T: (16, 6)
#   q @ k.T -> (4, 6)
#
# Expected StableHLO pattern:
#   transpose -> dot_general -> scalar scale -> broadcast -> divide
#
# Key observation:
#   Attention can be understood as a combination of reshape/transpose,
#   dot_general, scaling, softmax, and another dot_general.
# -----------------------------------------------------------------------------
def f_attention_score(q, k):
    d = q.shape[-1]
    return (q @ k.T) / jnp.sqrt(jnp.asarray(d, dtype=q.dtype))

q = jnp.ones((4, 16), dtype=jnp.float32)
k = jnp.ones((6, 16), dtype=jnp.float32)
inspect_jax(f_attention_score, q, k, name="Exercise 4: simplified attention score")

================================================================================Exercise 4: simplified attention score================================================================================
==== Jaxpr ====
{ lambda ; a:f32[4,16] b:f32[6,16]. let
    c:f32[16,6] = transpose[permutation=(1, 0)] b
    d:f32[4,6] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] a c
    e:f32[] = sqrt 16.0:f32[]
    f:f32[4,6] = div d e
  in (f,) }
==== StableHLO ====
module @jit_f_attention_score attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4x16xf32>, %arg1: tensor<6x16xf32>) -> (tensor<4x6xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.transpose %arg1, dims = [1, 0] : (tensor<6x16xf32>) -> tensor<16x6xf32>
    %1 = stablehlo.dot_general %arg0, %0, contracting_dims = [1] x [0], precision = [DEFAULT, DEFAULT] : (tensor<4x16xf32>, tensor<16x6xf32>) -> tensor<4x6xf32>
 

## 5. Reduction operations

Expected mapping:

```text
jnp.sum(x, axis=...)   → stablehlo.reduce
jnp.mean(x, axis=...)  → stablehlo.reduce + divide
jnp.max(x, axis=...)   → stablehlo.reduce with max combiner
```

In [14]:
# -----------------------------------------------------------------------------
# 5. Reduction operations
# -----------------------------------------------------------------------------
# This cell computes:
#
#   mean(x, axis=1)
#
# For x.shape == (3, 4), reducing axis=1 gives output shape (3,).
#
# Conceptually:
#   mean = reduce_sum / number_of_elements_along_axis
#
# Expected StableHLO pattern:
#   stablehlo.reduce with add combiner
#   divide by 4.0, often with scalar broadcast
#
# Key observation:
#   Reductions are central to loss functions, softmax, normalization, metrics,
#   log likelihoods, and aggregation.
# -----------------------------------------------------------------------------
def f_reduce(x):
    return jnp.mean(x, axis=1)

x5 = jnp.arange(12, dtype=jnp.float32).reshape(3, 4)
inspect_jax(f_reduce, x5, name="5. Reduction: mean over axis=1")

================================================================================5. Reduction: mean over axis=1================================================================================
==== Jaxpr ====
{ lambda ; a:f32[3,4]. let
    b:f32[3] = reduce_sum[axes=(1,)] a
    c:f32[3] = div b 4.0:f32[]
  in (c,) }
==== StableHLO ====
module @jit_f_reduce attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<3x4xf32>) -> (tensor<3xf32> {jax.result_info = "result"}) {
    %cst = stablehlo.constant dense<0.000000e+00> : tensor<f32>
    %0 = stablehlo.reduce(%arg0 init: %cst) applies stablehlo.add across dimensions = [1] : (tensor<3x4xf32>, tensor<f32>) -> tensor<3xf32>
    %cst_0 = stablehlo.constant dense<4.000000e+00> : tensor<f32>
    %1 = stablehlo.broadcast_in_dim %cst_0, dims = [] : (tensor<f32>) -> tensor<3xf32>
    %2 = stablehlo.divide %0, %1 : tensor<3xf32>
    return %2 : tensor<3xf32>
  }
}

==== Result ====
[1.5 5.5 9

### Exercise 5

Try `jnp.sum`, `jnp.max`, or `jax.nn.logsumexp`. `logsumexp` is especially useful for NumPyro-style models.

In [15]:
# -----------------------------------------------------------------------------
# Exercise 5: Custom reduction
# -----------------------------------------------------------------------------
# This exercise uses jax.nn.logsumexp(x, axis=1).
#
# logsumexp is typically implemented through numerically stable components:
#   max reduction
#   subtract max
#   exp
#   sum reduction
#   log
#   add max back
#
# Key observation:
#   High-level numerically stable functions often lower into multiple primitive
#   reductions and elementwise operations.
# -----------------------------------------------------------------------------
def f_reduce_exercise(x):
    # TODO: try sum, max, logsumexp
    return jax.nn.logsumexp(x, axis=1)

inspect_jax(f_reduce_exercise, x5, name="Exercise 5: custom reduction")

================================================================================Exercise 5: custom reduction================================================================================
==== Jaxpr ====
{ lambda ; a:f32[3,4]. let
    b:f32[3] = reduce_max[axes=(1,)] a
    c:f32[3] = max -inf:f32[] b
    d:bool[3] = is_finite c
    e:f32[3] = broadcast_in_dim[
      broadcast_dimensions=()
      shape=(3,)
      sharding=None
    ] 0.0:f32[]
    f:f32[3] = select_n d e c
    g:f32[3] = stop_gradient f
    h:f32[3,1] = broadcast_in_dim[
      broadcast_dimensions=(0,)
      shape=(3, 1)
      sharding=None
    ] g
    i:f32[3,4] = sub a h
    j:f32[3,4] = exp i
    k:f32[3] = reduce_sum[axes=(1,)] j
    _:f32[3] = sign k
    l:f32[3] = abs k
    m:f32[3] = log l
    n:f32[3] = add m g
  in (n,) }
==== StableHLO ====
module @jit_f_reduce_exercise attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<3x4xf32>) -> (tensor<3xf32> {

## 6. Conditional logic: `jnp.where` and `lax.cond`

Common mappings:

```text
jnp.where(cond, a, b) → stablehlo.select
lax.cond(...)         → compiler-visible control flow
```

In [16]:
# -----------------------------------------------------------------------------
# 6. Conditional logic: jnp.where and lax.cond
# -----------------------------------------------------------------------------
# This section compares two forms of conditional logic:
#
#   jnp.where:
#     elementwise value selection; usually lowers to stablehlo.select.
#
#   lax.cond:
#     compiler-visible branch/control flow; usually lowers to stablehlo.case.
#
# Key observation:
#   jnp.where is not a Python if statement. It selects values element by element.
#   lax.cond represents a branch in the compiled computation.
# -----------------------------------------------------------------------------
def f_where(x):
    return jnp.where(x > 0.0, x, -x)

x6 = jnp.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=jnp.float32)
inspect_jax(f_where, x6, name="6A. Elementwise conditional: jnp.where")


def f_cond(x):
    pred = jnp.sum(x) > 0.0
    return lax.cond(pred, lambda z: z * 2.0, lambda z: z - 2.0, x)

inspect_jax(f_cond, x6, name="6B. Control-flow conditional: lax.cond")

================================================================================6A. Elementwise conditional: jnp.where================================================================================
==== Jaxpr ====
{ lambda ; a:f32[5]. let
    b:bool[5] = gt a 0.0:f32[]
    c:f32[5] = neg a
    d:f32[5] = jit[
      name=_where
      jaxpr={ lambda ; b:bool[5] a:f32[5] c:f32[5]. let
          d:f32[5] = select_n b c a
        in (d,) }
    ] b a c
  in (d,) }
==== StableHLO ====
module @jit_f_where attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<5xf32>) -> (tensor<5xf32> {jax.result_info = "result"}) {
    %cst = stablehlo.constant dense<0.000000e+00> : tensor<f32>
    %0 = stablehlo.broadcast_in_dim %cst, dims = [] : (tensor<f32>) -> tensor<5xf32>
    %1 = stablehlo.compare  GT, %arg0, %0,  FLOAT : (tensor<5xf32>, tensor<5xf32>) -> tensor<5xi1>
    %2 = stablehlo.negate %arg0 : tensor<5xf32>
    %3 = call @_where(%1, %ar

## 7. Loops: Python loop vs `lax.scan`

A Python loop inside `jit` can be unrolled when the loop count is static. `lax.scan` creates a compiler-visible loop and is usually better for long recurrent computations.

In [17]:
# -----------------------------------------------------------------------------
# 7. Loops: Python loop vs lax.scan
# -----------------------------------------------------------------------------
# This section compares:
#
#   Python loop with static range:
#     Often unrolled during tracing, so repeated operations appear many times in
#     Jaxpr/StableHLO.
#
#   lax.scan:
#     Represents a recurrent computation as a compiler-visible loop, typically
#     lowering to stablehlo.while.
#
# Key observation:
#   For long recurrent computations, state-space models, RNNs, rollouts, and
#   simulations, lax.scan is usually preferable to a large unrolled Python loop.
# -----------------------------------------------------------------------------
def f_python_loop(x):
    y = x
    for _ in range(5):
        y = y * 1.1 + 1.0
    return y

x7 = jnp.ones((3,), dtype=jnp.float32)
inspect_jax(f_python_loop, x7, name="7A. Python loop with static range")


def f_scan(xs):
    def body(carry, x):
        carry = carry * 1.1 + x
        return carry, carry

    init = 0.0
    carry, ys = lax.scan(body, init, xs)
    return ys

xs7 = jnp.ones((5,), dtype=jnp.float32)
inspect_jax(f_scan, xs7, name="7B. lax.scan")

================================================================================7A. Python loop with static range================================================================================
==== Jaxpr ====
{ lambda ; a:f32[3]. let
    b:f32[3] = mul a 1.100000023841858:f32[]
    c:f32[3] = add b 1.0:f32[]
    d:f32[3] = mul c 1.100000023841858:f32[]
    e:f32[3] = add d 1.0:f32[]
    f:f32[3] = mul e 1.100000023841858:f32[]
    g:f32[3] = add f 1.0:f32[]
    h:f32[3] = mul g 1.100000023841858:f32[]
    i:f32[3] = add h 1.0:f32[]
    j:f32[3] = mul i 1.100000023841858:f32[]
    k:f32[3] = add j 1.0:f32[]
  in (k,) }
==== StableHLO ====
module @jit_f_python_loop attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<3xf32>) -> (tensor<3xf32> {jax.result_info = "result"}) {
    %cst = stablehlo.constant dense<1.100000e+00> : tensor<f32>
    %0 = stablehlo.broadcast_in_dim %cst, dims = [] : (tensor<f32>) -> tensor<3xf32>
    %1 

### Exercise 7

Increase the Python loop length and compare with `lax.scan`. Which version scales better for 100, 1,000, or 10,000 steps?

In [18]:
# -----------------------------------------------------------------------------
# Exercise 7: Nonlinear scan recurrence
# -----------------------------------------------------------------------------
# This exercise changes the scan body to:
#
#   carry_{t+1} = tanh(carry_t + x_t)
#
# The scan returns:
#   final carry
#   all intermediate carries as ys
#
# Expected StableHLO pattern:
#   stablehlo.while
#   dynamic_slice
#   add
#   tanh
#   dynamic_update_slice
#
# Key observation:
#   This is a minimal RNN/state-space style recurrence expressed in a form that
#   JAX can lower as loop IR.
# -----------------------------------------------------------------------------
def f_scan_exercise(xs):
    def body(carry, x):
        # TODO: modify recurrence
        carry = jnp.tanh(carry + x)
        return carry, carry

    carry, ys = lax.scan(body, 0.0, xs)
    return carry, ys

inspect_jax(f_scan_exercise, xs7, name="Exercise 7: custom scan")

================================================================================Exercise 7: custom scan================================================================================
==== Jaxpr ====
{ lambda ; a:f32[5]. let
    b:f32[] c:f32[5] = scan[
      _split_transpose=False
      jaxpr={ lambda ; d:f32[] e:f32[]. let
          f:f32[] = convert_element_type[new_dtype=float32 weak_type=False] d
          g:f32[] = add f e
          h:f32[] = tanh g
        in (h, h) }
      length=5
      linear=(False, False)
      num_carry=1
      num_consts=0
      reverse=False
      unroll=1
    ] 0.0:f32[] a
  in (b, c) }
==== StableHLO ====
module @jit_f_scan_exercise attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<5xf32>) -> (tensor<f32> {jax.result_info = "result[0]"}, tensor<5xf32> {jax.result_info = "result[1]"}) {
    %cst = stablehlo.constant dense<0.000000e+00> : tensor<f32>
    %cst_0 = stablehlo.constant dense<0.00

## 8. Automatic differentiation: `grad`

`jax.grad` transforms the computation into another computation that evaluates derivatives. Inspect the additional operations generated by the backward pass.

In [19]:
# -----------------------------------------------------------------------------
# 8. Automatic differentiation: grad
# -----------------------------------------------------------------------------
# This cell computes the gradient of a scalar MSE loss with respect to w:
#
#   loss = mean((x @ w - y)^2)
#   grad_w = d loss / d w
#
# For this linear model:
#   grad_w = x.T @ ((2 / N) * (x @ w - y))
#
# Key observations:
#   1. grad is a program transformation, not finite-difference numerical probing.
#   2. Jaxpr may show parts of the forward loss calculation.
#   3. StableHLO may remove the scalar loss itself if only the gradient is needed.
# -----------------------------------------------------------------------------
def loss_fn(w, x, y):
    pred = x @ w
    return jnp.mean((pred - y) ** 2)

grad_loss_fn = jax.grad(loss_fn)

w8 = jnp.ones((16, 1), dtype=jnp.float32)
x8 = jnp.ones((8, 16), dtype=jnp.float32)
y8 = jnp.ones((8, 1), dtype=jnp.float32)
inspect_jax(grad_loss_fn, w8, x8, y8, name="8. grad(loss_fn)")

================================================================================8. grad(loss_fn)================================================================================
==== Jaxpr ====
{ lambda ; a:f32[16,1] b:f32[8,16] c:f32[8,1]. let
    d:f32[8,1] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] b a
    e:f32[8,1] = sub d c
    f:f32[8,1] = integer_pow[y=2] e
    g:f32[8,1] = integer_pow[y=1] e
    h:f32[8,1] = mul 2.0:f32[] g
    i:f32[] = reduce_sum[axes=(0, 1)] f
    _:f32[] = div i 8.0:f32[]
    j:f32[] = div 1.0:f32[] 8.0:f32[]
    k:f32[8,1] = broadcast_in_dim[
      broadcast_dimensions=()
      shape=(8, 1)
      sharding=None
    ] j
    l:f32[8,1] = mul k h
    m:f32[1,16] = dot_general[
      dimension_numbers=(([0], [0]), ([], []))
      preferred_element_type=float32
    ] l b
    n:f32[16,1] = transpose[permutation=(1, 0)] m
  in (n,) }
==== StableHLO ====
module @jit_loss_fn attributes {mhlo.num_partition

### Exercise 8

Modify the loss: L1 loss, Huber-like loss using `jnp.where`, or logistic loss. Then inspect how the gradient changes.

In [20]:
# -----------------------------------------------------------------------------
# Exercise 8: Custom gradient with L1 loss
# -----------------------------------------------------------------------------
# This exercise changes the loss from MSE to L1:
#
#   loss = mean(abs(x @ w - y))
#
# Conceptual gradient:
#   grad_w = x.T @ ((1 / N) * sign(x @ w - y))
#
# Compared with MSE:
#   MSE gradient uses 2 * err.
#   L1 gradient uses sign(err), with special handling near zero.
#
# Key observation:
#   Changing the loss changes the backward graph generated by jax.grad.
# -----------------------------------------------------------------------------
def custom_loss_fn(w, x, y):
    pred = x @ w
    err = pred - y
    # TODO: modify the loss
    return jnp.mean(jnp.abs(err))

custom_grad_loss_fn = jax.grad(custom_loss_fn)
inspect_jax(custom_grad_loss_fn, w8, x8, y8, name="Exercise 8: custom gradient")

================================================================================Exercise 8: custom gradient================================================================================
==== Jaxpr ====
{ lambda ; a:f32[16,1] b:f32[8,16] c:f32[8,1]. let
    d:f32[8,1] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] b a
    e:f32[8,1] = sub d c
    f:f32[8,1] = abs e
    g:bool[8,1] = ge e 0.0:f32[]
    h:f32[] = reduce_sum[axes=(0, 1)] f
    _:f32[] = div h 8.0:f32[]
    i:f32[] = div 1.0:f32[] 8.0:f32[]
    j:f32[8,1] = broadcast_in_dim[
      broadcast_dimensions=()
      shape=(8, 1)
      sharding=None
    ] i
    k:f32[8,1] = broadcast_in_dim[
      broadcast_dimensions=()
      shape=(8, 1)
      sharding=None
    ] 0.0:f32[]
    l:f32[8,1] = select_n g j k
    m:f32[8,1] = select_n g k j
    n:f32[8,1] = neg l
    o:f32[8,1] = add_any m n
    p:f32[1,16] = dot_general[
      dimension_numbers=(([0], [0]), ([], []))
      

## 9. Batching: `vmap`

`vmap` transforms a function so that operations themselves become batched. It is not merely a faster Python loop.

In [21]:
# -----------------------------------------------------------------------------
# 9. Batching: vmap
# -----------------------------------------------------------------------------
# This cell maps a single-example dot product over a batch:
#
#   single:  (16,) dot (16,) -> scalar
#   batched: (8,16) dot (16,) -> (8,)
#
# vmap is not a runtime loop operation. It is a function transformation that turns
# a single-example computation into a batched computation.
#
# Key observation:
#   vmap itself usually disappears from StableHLO; the transformed batched
#   primitive, here dot_general, remains.
# -----------------------------------------------------------------------------
def single_dot(x, w):
    return jnp.dot(x, w)

batched_dot = jax.vmap(single_dot, in_axes=(0, None))

x9 = jnp.ones((8, 16), dtype=jnp.float32)
w9 = jnp.ones((16,), dtype=jnp.float32)
inspect_jax(batched_dot, x9, w9, name="9. vmap(single_dot)")

================================================================================9. vmap(single_dot)================================================================================
==== Jaxpr ====
{ lambda ; a:f32[8,16] b:f32[16]. let
    c:f32[8] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] a b
  in (c,) }
==== StableHLO ====
module @jit_single_dot attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<8x16xf32>, %arg1: tensor<16xf32>) -> (tensor<8xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.dot_general %arg0, %arg1, contracting_dims = [1] x [0], precision = [DEFAULT, DEFAULT] : (tensor<8x16xf32>, tensor<16xf32>) -> tensor<8xf32>
    return %0 : tensor<8xf32>
  }
}

==== Result ====
[16. 16. 16. 16. 16. 16. 16. 16.]


### Exercise 9

Try `vmap(grad(f))`, which is important for per-sample gradients, influence estimation, DP-SGD, and Bayesian workflows.

In [22]:
# -----------------------------------------------------------------------------
# Exercise 9: vmap(grad(loss_per_example))
# -----------------------------------------------------------------------------
# This exercise computes per-example gradients:
#
#   loss_i = (dot(x_i, w) - y_i)^2
#   grad_i = d loss_i / d w
#
# Output shape:
#   (batch, parameter_dim) = (8, 16)
#
# This is the first step used in DP-SGD:
#   1. compute per-example gradients
#   2. clip each gradient by norm
#   3. average clipped gradients
#   4. add Gaussian noise
#   5. apply optimizer update
#
# Key observation:
#   grad(batch_loss) returns one aggregated gradient.
#   vmap(grad(loss_per_example)) returns one gradient per sample.
# -----------------------------------------------------------------------------
def loss_per_example(w, x, y):
    pred = jnp.dot(x, w)
    return (pred - y) ** 2

per_example_grad = jax.vmap(jax.grad(loss_per_example), in_axes=(None, 0, 0))

w9b = jnp.ones((16,), dtype=jnp.float32)
x9b = jnp.ones((8, 16), dtype=jnp.float32)
y9b = jnp.ones((8,), dtype=jnp.float32)
inspect_jax(per_example_grad, w9b, x9b, y9b, name="Exercise 9: vmap(grad(loss_per_example))")

================================================================================Exercise 9: vmap(grad(loss_per_example))================================================================================
==== Jaxpr ====
{ lambda ; a:f32[16] b:f32[8,16] c:f32[8]. let
    d:f32[8] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] b a
    e:f32[8] = sub d c
    _:f32[8] = integer_pow[y=2] e
    f:f32[8] = integer_pow[y=1] e
    g:f32[8] = mul 2.0:f32[] f
    h:f32[8] = mul 1.0:f32[] g
    i:f32[8,16] = dot_general[
      dimension_numbers=(([], []), ([0], [0]))
      preferred_element_type=float32
    ] h b
  in (i,) }
==== StableHLO ====
module @jit_loss_per_example attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<16xf32>, %arg1: tensor<8x16xf32>, %arg2: tensor<8xf32>) -> (tensor<8x16xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.dot_general %arg1, %arg0, contract

## 10. Gather and scatter: GNN-style operations

GNNs frequently use indexed edge operations and aggregation:

```text
node_features[src]       → gather-like operation
out.at[dst].add(messages) → scatter-like operation
```

These operations can be performance-sensitive on GPU.

In [24]:
# -----------------------------------------------------------------------------
# 10. Gather and scatter: GNN-style operations
# -----------------------------------------------------------------------------
# This cell implements a minimal GNN message-passing aggregation:
#
#   messages = node_features[src]        # gather source node features
#   out[dst] += messages                 # scatter-add into destination nodes
#
# Mathematical form:
#   out[v] = sum_{e: dst[e] = v} node_features[src[e]]
#
# Expected StableHLO pattern:
#   gather
#   scatter with add combiner
#
# Key observation:
#   GNN message passing often lowers to irregular gather/scatter operations,
#   unlike dense neural network layers that often lower to dot_general.
# -----------------------------------------------------------------------------
def gnn_message_sum(node_features, src, dst, num_nodes):
    messages = node_features[src]
    out = jnp.zeros((num_nodes, node_features.shape[1]), dtype=node_features.dtype)
    out = out.at[dst].add(messages)
    return out

node_features = jnp.arange(5 * 3, dtype=jnp.float32).reshape(5, 3)
src = jnp.array([0, 1, 2, 3, 1, 4], dtype=jnp.int32)
dst = jnp.array([1, 2, 3, 4, 4, 0], dtype=jnp.int32)
num_nodes = 5

gnn_jitted = jax.jit(gnn_message_sum, static_argnums=(3,))

print("==== Jaxpr ====")
print(jax.make_jaxpr(lambda nf, s, d: gnn_message_sum(nf, s, d, num_nodes))(node_features, src, dst))
print("==== StableHLO ====")
print(gnn_jitted.lower(node_features, src, dst, num_nodes).compiler_ir(dialect="stablehlo"))
print("==== Result ====")
print(gnn_jitted(node_features, src, dst, num_nodes))

==== Jaxpr ====
{ lambda ; a:f32[5,3] b:i32[6] c:i32[6]. let
    d:bool[6] = lt b 0:i32[]
    e:i32[6] = add b 5:i32[]
    f:i32[6] = select_n d b e
    g:i32[6,1] = broadcast_in_dim[
      broadcast_dimensions=(0,)
      shape=(6, 1)
      sharding=None
    ] f
    h:f32[6,3] = gather[
      dimension_numbers=GatherDimensionNumbers(offset_dims=(1,), collapsed_slice_dims=(0,), start_index_map=(0,), operand_batching_dims=(), start_indices_batching_dims=())
      fill_value=None
      indices_are_sorted=False
      mode=GatherScatterMode.PROMISE_IN_BOUNDS
      slice_sizes=(1, 3)
      unique_indices=False
    ] a g
    i:f32[5,3] = broadcast_in_dim[
      broadcast_dimensions=()
      shape=(5, 3)
      sharding=None
    ] 0.0:f32[]
    j:bool[6] = lt c 0:i32[]
    k:i32[6] = add c 5:i32[]
    l:i32[6] = select_n j c k
    m:i32[6,1] = broadcast_in_dim[
      broadcast_dimensions=(0,)
      shape=(6, 1)
      sharding=None
    ] l
    n:f32[5,3] = scatter-add[
      dimension_numbers=Sc

### Exercise 10

Add edge weights and inspect whether broadcasting appears before scatter.

In [25]:
# -----------------------------------------------------------------------------
# Exercise 10: Edge-weighted GNN message passing
# -----------------------------------------------------------------------------
# This exercise adds edge weights to the message:
#
#   messages[e] = edge_weight[e] * node_features[src[e]]
#   out[dst[e]] += messages[e]
#
# Shape flow:
#   node_features[src] : (num_edges, feature_dim)
#   edge_weight        : (num_edges,)
#   edge_weight[:,None]: (num_edges, 1)
#   broadcast          : (num_edges, feature_dim)
#
# Expected StableHLO pattern:
#   gather -> broadcast -> multiply -> scatter-add
#
# Key observation:
#   This resembles weighted message passing, GCN normalization, or attention
#   coefficients in graph attention networks.
# -----------------------------------------------------------------------------
def weighted_gnn_message_sum(node_features, src, dst, edge_weight, num_nodes):
    messages = node_features[src] * edge_weight[:, None]
    out = jnp.zeros((num_nodes, node_features.shape[1]), dtype=node_features.dtype)
    return out.at[dst].add(messages)

edge_weight = jnp.ones((src.shape[0],), dtype=jnp.float32)
weighted_gnn_jitted = jax.jit(weighted_gnn_message_sum, static_argnums=(4,))
print(weighted_gnn_jitted.lower(node_features, src, dst, edge_weight, num_nodes).compiler_ir(dialect="stablehlo"))
print(weighted_gnn_jitted(node_features, src, dst, edge_weight, num_nodes))

module @jit_weighted_gnn_message_sum attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<5x3xf32>, %arg1: tensor<6xi32>, %arg2: tensor<6xi32>, %arg3: tensor<6xf32>) -> (tensor<5x3xf32> {jax.result_info = "result"}) {
    %c = stablehlo.constant dense<0> : tensor<i32>
    %0 = stablehlo.broadcast_in_dim %c, dims = [] : (tensor<i32>) -> tensor<6xi32>
    %1 = stablehlo.compare  LT, %arg1, %0,  SIGNED : (tensor<6xi32>, tensor<6xi32>) -> tensor<6xi1>
    %c_0 = stablehlo.constant dense<5> : tensor<i32>
    %2 = stablehlo.broadcast_in_dim %c_0, dims = [] : (tensor<i32>) -> tensor<6xi32>
    %3 = stablehlo.add %arg1, %2 : tensor<6xi32>
    %4 = stablehlo.select %1, %3, %arg1 : tensor<6xi1>, tensor<6xi32>
    %5 = stablehlo.broadcast_in_dim %4, dims = [0] : (tensor<6xi32>) -> tensor<6x1xi32>
    %6 = "stablehlo.gather"(%arg0, %5) <{dimension_numbers = #stablehlo.gather<offset_dims = [1], collapsed_slice_dims = [0], start_index_map =

## 11. Static arguments and recompilation risk

JAX compilation depends on shapes, dtypes, and static arguments. If an argument controls output shape, it often needs to be static.

In [26]:
# -----------------------------------------------------------------------------
# 11. Static arguments and recompilation risk
# -----------------------------------------------------------------------------
# This cell illustrates why values used to define array shapes often need to be
# static at compile time.
#
# n controls the shape of jnp.arange(n). Therefore n is marked static.
#
# Key observation:
#   Changing a static argument changes the compiled program shape and can trigger
#   recompilation.
# -----------------------------------------------------------------------------
def make_range_scaled(x, n):
    r = jnp.arange(n, dtype=x.dtype)
    return x + r

x11 = jnp.array(1.0, dtype=jnp.float32)

try:
    print(jax.jit(make_range_scaled)(x11, 4))
except Exception as e:
    print("Expected issue when n is not static:")
    print(repr(e))

make_range_scaled_jit = jax.jit(make_range_scaled, static_argnums=(1,))
print(make_range_scaled_jit(x11, 4))
print(make_range_scaled_jit.lower(x11, 4).compiler_ir(dialect="stablehlo"))

for n in [4, 8, 16]:
    print("n =", n, "result =", make_range_scaled_jit(x11, n))

Expected issue when n is not static:
ConcretizationTypeError("Abstract tracer value encountered where concrete value is expected: traced array with shape int32[]\nIt arose in the jnp.arange argument 'stop'\nThe error occurred while tracing the function make_range_scaled at /tmp/ipykernel_11319/1653866984.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument n.\n\nSee https://docs.jax.dev/en/latest/errors.html#jax.errors.ConcretizationTypeError")
[1. 2. 3. 4.]
module @jit_make_range_scaled attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<f32>) -> (tensor<4xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.iota dim = 0 : tensor<4xf32>
    %1 = stablehlo.broadcast_in_dim %arg0, dims = [] : (tensor<f32>) -> tensor<4xf32>
    %2 = stablehlo.add %1, %0 : tensor<4xf32>
    return %2 : tensor<4xf32>
  }
}

n = 4 result = [1. 2. 3. 4.]
n = 8 result = [1. 2. 3. 4. 5. 6. 7

## 12. Mini case study: Bayesian-style log probability

A lot of NumPyro-like computation is tensor algebra plus reduction:

```text
linear algebra
broadcasting
log / exp
reduction
```

In [27]:
# -----------------------------------------------------------------------------
# 12. Mini case study: Bayesian-style log probability
# -----------------------------------------------------------------------------
# This cell computes a Gaussian log likelihood:
#
#   mu = x @ beta
#   resid = y - mu
#   logp = Gaussian log density
#   return sum(logp)
#
# Expected StableHLO pattern:
#   dot_general
#   subtract
#   divide
#   power/multiply
#   log
#   reduce
#
# Key observation:
#   Probabilistic modeling code often lowers to a combination of dense linear
#   algebra, elementwise log-density terms, and reductions.
# -----------------------------------------------------------------------------
def gaussian_log_likelihood(beta, x, y, sigma):
    mu = x @ beta
    resid = y - mu
    logp = -0.5 * (resid / sigma) ** 2 - jnp.log(sigma) - 0.5 * jnp.log(2.0 * jnp.pi)
    return jnp.sum(logp)

beta12 = jnp.ones((4,), dtype=jnp.float32)
x12 = jnp.ones((10, 4), dtype=jnp.float32)
y12 = jnp.ones((10,), dtype=jnp.float32)
sigma12 = jnp.array(1.5, dtype=jnp.float32)
inspect_jax(gaussian_log_likelihood, beta12, x12, y12, sigma12, name="12. Gaussian log likelihood")

================================================================================12. Gaussian log likelihood================================================================================
==== Jaxpr ====
{ lambda ; a:f32[4] b:f32[10,4] c:f32[10] d:f32[]. let
    e:f32[10] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] b a
    f:f32[10] = sub c e
    g:f32[10] = div f d
    h:f32[10] = integer_pow[y=2] g
    i:f32[10] = mul -0.5:f32[] h
    j:f32[] = log d
    k:f32[10] = sub i j
    l:f32[] = log 6.283185307179586:f32[]
    m:f32[] = mul 0.5:f32[] l
    n:f32[] = convert_element_type[new_dtype=float32 weak_type=False] m
    o:f32[10] = sub k n
    p:f32[] = reduce_sum[axes=(0,)] o
  in (p,) }
==== StableHLO ====
module @jit_gaussian_log_likelihood attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4xf32>, %arg1: tensor<10x4xf32>, %arg2: tensor<10xf32>, %arg3: tensor<f

### Exercise 12

Extend the model by adding intercept, per-observation sigma, prior log probability, or negative log posterior.

In [28]:
# -----------------------------------------------------------------------------
# Exercise 12: Gaussian log posterior
# -----------------------------------------------------------------------------
# This exercise extends the likelihood with an intercept and simple Gaussian
# priors over beta and intercept.
#
# Conceptual structure:
#   log posterior = log likelihood + log prior(beta) + log prior(intercept)
#
# Expected StableHLO pattern:
#   dot_general
#   broadcast/add for intercept
#   elementwise log-density terms
#   reductions over observations and parameters
#
# Key observation:
#   Bayesian-style objectives are usually compiler-friendly when written with
#   vectorized JAX primitives and explicit reductions.
# -----------------------------------------------------------------------------
def gaussian_log_posterior_exercise(beta, intercept, x, y, sigma):
    mu = x @ beta + intercept
    resid = y - mu
    log_lik = -0.5 * (resid / sigma) ** 2 - jnp.log(sigma) - 0.5 * jnp.log(2.0 * jnp.pi)
    log_prior_beta = -0.5 * jnp.sum(beta ** 2)
    return jnp.sum(log_lik) + log_prior_beta

intercept12 = jnp.array(0.1, dtype=jnp.float32)
inspect_jax(
    gaussian_log_posterior_exercise,
    beta12,
    intercept12,
    x12,
    y12,
    sigma12,
    name="Exercise 12: Gaussian log posterior",
)

================================================================================Exercise 12: Gaussian log posterior================================================================================
==== Jaxpr ====
{ lambda ; a:f32[4] b:f32[] c:f32[10,4] d:f32[10] e:f32[]. let
    f:f32[10] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] c a
    g:f32[10] = add f b
    h:f32[10] = sub d g
    i:f32[10] = div h e
    j:f32[10] = integer_pow[y=2] i
    k:f32[10] = mul -0.5:f32[] j
    l:f32[] = log e
    m:f32[10] = sub k l
    n:f32[] = log 6.283185307179586:f32[]
    o:f32[] = mul 0.5:f32[] n
    p:f32[] = convert_element_type[new_dtype=float32 weak_type=False] o
    q:f32[10] = sub m p
    r:f32[4] = integer_pow[y=2] a
    s:f32[] = reduce_sum[axes=(0,)] r
    t:f32[] = mul -0.5:f32[] s
    u:f32[] = reduce_sum[axes=(0,)] q
    v:f32[] = add u t
  in (v,) }
==== StableHLO ====
module @jit_gaussian_log_posterior_exercise attributes 

## 13. Reading checklist

When you inspect StableHLO, use this checklist:

```text
1. Are the expected high-level operations present?
   - dot_general
   - reduce
   - gather
   - scatter
   - while

2. Are there many shape transformations?
   - reshape
   - transpose
   - broadcast_in_dim

3. Is a Python loop unrolled?
   - many repeated operations may indicate unrolling

4. Are shape-controlling values static?
   - static_argnums may be needed

5. Are gather/scatter operations central?
   - common in GNNs, sparse workflows, indexing-heavy code

6. Could changing input shapes trigger recompilation?
   - JAX compiles per shape/dtype/static-arg signature
```

## 14. Suggested next practice

After finishing this notebook, apply the same inspection workflow to your own code:

```text
1. A small GNN message passing function
2. A NumPyro log_prob-like function
3. A Transformer attention block
4. A time-series recurrence with lax.scan
5. A vmap + grad function
```

The target skill is:

```text
Given JAX code, predict the rough Jaxpr and StableHLO structure before printing it.
```

That is the practical bridge from writing JAX to understanding how XLA sees your program.